In [22]:
import torch
from torch import nn

In [81]:
class LinearRegression(nn.Module):

    def __init__(self, lr):
        super().__init__()
        self.lr = lr
        self.net = nn.LazyLinear(1)
        self.net.weight.data.normal_(0, 0.01)
        self.net.bias.data.fill_(0)

    def forward(self, X):
        return self.net(X)

    def loss(self, y_hat, y):
        fn = nn.MSELoss()
        return fn(y_hat, y)

    def config_optimizers(self):
        return torch.optim.SGD(self.parameters(), self.lr)


In [82]:
class SyntheticData():
    def __init__(self, w, b, noise = 0.01,batch_size = 32, num_of_training_data = 10000, num_of_validation_data = 10000, device = "cpu"):
        self.w = w
        self.b = b
        self.batch_size = batch_size
        self.num_of_training_data = num_of_training_data
        self.num_of_validation_data = num_of_validation_data
        self.device = device
        self.n = self.num_of_training_data + self.num_of_validation_data
        self.X = torch.randn((self.n, len(self.w)))
        self.noise = torch.randn(self.n,1) * noise
        self.Y = torch.matmul(self.X, self.w.reshape(-1,1)) + self.b + self.noise

    def get_dataloader(self, train):
        indicies = slice(0,self.num_of_training_data) if train else slice(self.num_of_training_data, None)
        dataset = torch.utils.data.TensorDataset(self.X[indicies], self.Y[indicies])
        return torch.utils.data.DataLoader(dataset,batch_size=self.batch_size, shuffle = train)

    def training_data(self):
        return self.get_dataloader(True)


In [83]:
def Training(data, model, optim):
    model.train()

    for (X, y) in data.training_data():

        pred = model(X)
        loss = model.loss(pred, y)

        loss.backward()
        optim.step()
        optim.zero_grad()


In [87]:
data = SyntheticData(torch.tensor([2,-3.4]), 5)
print(data.X[:3], data.Y[:3])
model = LinearRegression(0.03)
optim = model.config_optimizers()
for epoch in range(10):
    Training(data, model, optim)

tensor([[ 2.2685,  1.7695],
        [ 0.4064,  0.0324],
        [-0.2421,  0.3933]]) tensor([[3.5308],
        [5.6853],
        [3.1790]])


In [88]:
model.net.weight

Parameter containing:
tensor([[ 1.9997, -3.3995]], requires_grad=True)

In [90]:
model(torch.tensor( (2.2685,  1.7695)))

tensor([3.5208], grad_fn=<ViewBackward0>)